# Session 08 · Web Scraping in Python

Run each cell with **Shift+Enter**. Original material by Aseef Ahmed.

In [1]:
# Setup: run from the code/ folder so local modules and data files are found
import os
if os.path.isdir('code'):
    os.chdir('code')

## Example 1 - parse and find

In [2]:
import sys
from bs4 import BeautifulSoup

html = open("sample_store.html", encoding="utf-8").read()
soup = BeautifulSoup(html, "html.parser")
print(soup.title.get_text())
print(soup.find("h1").get_text(strip=True))
print(len(soup.find_all("div", class_="product")), "products")

Gadget Bazaar - practice store
Gadget Bazaar
8 products


## Example 2 - extract every product

In [3]:
products = []
for card in soup.select("div.product"):
    products.append({
        "title": card.select_one(".title").get_text(strip=True),
        "price": int(card.select_one(".price").get_text().replace("Rs", "").replace(",", "")),
        "in_stock": "in" in card.select_one(".stock")["class"],
        "category": card["data-category"],
    })
for p in products[:3]:
    print(p)

{'title': 'Wireless Mouse', 'price': 1850, 'in_stock': True, 'category': 'mice'}
{'title': 'Gaming Mouse RGB', 'price': 4200, 'in_stock': False, 'category': 'mice'}
{'title': 'Mechanical Keyboard', 'price': 8999, 'in_stock': True, 'category': 'keyboards'}


## Into pandas and CSV

In [4]:
import pandas as pd
df = pd.DataFrame(products)
print(df.groupby("category")["price"].mean().round(0))

category
accessories    4425.0
keyboards      5724.0
mice           3025.0
storage        6400.0
Name: price, dtype: float64


## Example 4 - robots.txt rules

In [5]:
from urllib import robotparser
rp = robotparser.RobotFileParser()
rp.parse(["User-agent: *", "Disallow: /admin/"])
print(rp.can_fetch("*", "https://example.com/admin/"), rp.can_fetch("*", "https://example.com/products"))

False True


## Example 3 - live, polite download (optional)
Set `LIVE = True` in the notebook, or run the script with `--live`.

In [6]:
LIVE = "--live" in sys.argv
if LIVE:
    import time
    import requests
    HEADERS = {"User-Agent": "DiplomaAI-student-scraper/1.0 (learning project)"}
    r = requests.get("https://books.toscrape.com/", headers=HEADERS, timeout=10)
    r.raise_for_status()
    r.encoding = r.apparent_encoding  # fixes symbols like £ showing as Â£
    live = BeautifulSoup(r.text, "html.parser")
    for book in live.select("article.product_pod")[:5]:
        print(book.h3.a["title"], "|", book.select_one(".price_color").get_text())
    time.sleep(1)
else:
    print("Live demo skipped (offline mode).")

Live demo skipped (offline mode).
